# Amazon Product Data Analysis and Price Prediction

## 1. Introduction

This Jupyter notebook presents a comprehensive machine learning pipeline for analyzing Amazon product data and predicting product discounted prices. The dataset contains various details about products, including their names, categories, pricing, ratings, reviews, and other descriptive information.

The main objectives of this notebook are:
1.  **Understand the Data:** Perform Exploratory Data Analysis (EDA) to gain insights into the dataset's structure, distributions, and relationships between features.
2.  **Data Preprocessing:** Clean and transform raw data into a suitable format for machine learning models, handling missing values, outliers, and converting data types.
3.  **Feature Engineering:** Create new features from existing ones to enhance model performance.
4.  **Model Building:** Develop a regression model to predict the `discounted_price` of products.
5.  **Model Evaluation:** Assess the performance of the developed model using appropriate metrics and visualizations.
6.  **Hyperparameter Tuning:** Optimize model parameters for improved accuracy and generalization.
7.  **Insights and Conclusion:** Summarize key findings and suggest future improvements.

### Dataset Schema Overview:

*   `product_id`: Unique identifier for each product.
*   `product_name`: Name of the product.
*   `category`: Hierarchical category path of the product.
*   `discounted_price`: The price after discount. (Target variable for regression)
*   `actual_price`: The original price of the product.
*   `discount_percentage`: The percentage discount offered.
*   `rating`: Average rating of the product.
*   `rating_count`: Number of ratings received.
*   `about_product`: Description of the product.
*   `user_id`: Comma-separated list of user IDs who reviewed the product.
*   `user_name`: Comma-separated list of user names who reviewed the product.
*   `review_id`: Comma-separated list of review IDs.
*   `review_title`: Comma-separated list of review titles.
*   `review_content`: Comma-separated list of review contents.
*   `img_link`: URL to the product image.
*   `product_link`: URL to the product page on Amazon.

Our goal is to predict `discounted_price` using other relevant features. This will be a regression task.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import logging
import os
import re
import pickle

# Create directories for logs and artifacts
log_dir = 'ml_logs'
artifact_dir = 'artifacts'
os.makedirs(log_dir, exist_ok=True)
os.makedirs(artifact_dir, exist_ok=True)

# Configure logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[
                        logging.FileHandler(os.path.join(log_dir, 'ml_pipeline.log')),
                        logging.StreamHandler()
                    ])

logging.info("Starting Amazon Product Data Analysis and Price Prediction Pipeline.")

# Set plot style
sns.set_style("whitegrid")


## 2. Data Loading

In this section, we will load the dataset from the specified CSV file (`data/amazon_products.csv`). We'll implement error handling to manage potential `FileNotFoundError` if the dataset is not present in the expected location. After loading, we'll display the first few rows, check the data types, and get a concise summary of the dataset.

**Note:** For the purpose of this notebook, we are assuming the presence of `data/amazon_products.csv`. If you are running this, ensure the file is in the `data/` directory relative to your notebook.


In [ ]:
DATA_PATH = 'data/amazon_products.csv'

try:
    df = pd.read_csv(DATA_PATH)
    logging.info(f"Successfully loaded data from {DATA_PATH}. Shape: {df.shape}")
    print("Dataset loaded successfully. First 5 rows:")
    print(df.head())
    print("\nDataset Info:")
    df.info()
except FileNotFoundError:
    logging.error(f"Error: The file '{DATA_PATH}' was not found. Please ensure the dataset is in the correct directory.")
    print(f"Error: The file '{DATA_PATH}' was not found. Please ensure the dataset is in the correct directory.")
    # Exit or create a dummy dataframe to continue with the notebook structure for demonstration
    df = pd.DataFrame() # Create an empty DataFrame to prevent further errors
except Exception as e:
    logging.error(f"An unexpected error occurred during data loading: {e}")
    print(f"An unexpected error occurred during data loading: {e}")
    df = pd.DataFrame() # Create an empty DataFrame



## 3. Exploratory Data Analysis (EDA)

Exploratory Data Analysis is a crucial step to understand the data's characteristics, identify patterns, and detect anomalies. In this section, we will:
*   Check for missing values across all columns.
*   Identify and correct data types for numerical columns, especially those represented as objects (strings) due to special characters (e.g., '₹', ',', '%').
*   Perform feature engineering to extract meaningful insights from existing text and multi-valued columns.
*   Analyze unique values and distributions of key features.
*   Detect and handle duplicate entries if any.

### Missing Values Check


In [ ]:
if not df.empty:
    logging.info("Checking for missing values.")
    missing_values = df.isnull().sum()
    missing_percentage = (df.isnull().sum() / len(df)) * 100
    missing_info = pd.DataFrame({'Missing Count': missing_values, 'Percentage': missing_percentage})
    print("\nMissing Values Information:")
    print(missing_info[missing_info['Missing Count'] > 0].sort_values(by='Percentage', ascending=False))
else:
    logging.warning("DataFrame is empty, skipping missing values check.")


### Data Type Conversion and Cleaning

Many columns like `discounted_price`, `actual_price`, `discount_percentage`, `rating`, and `rating_count` are currently objects (strings). They contain special characters like currency symbols, commas, and percentage signs that need to be removed before converting them to numerical types.


In [ ]:
if not df.empty:
    logging.info("Performing data type conversion and cleaning.")
    
    # Handle currency and comma for price columns
    for col in ['discounted_price', 'actual_price']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('₹', '', regex=False).str.replace(',', '', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')
            logging.info(f"Cleaned and converted '{col}' to numeric.")

    # Handle percentage for discount_percentage
    if 'discount_percentage' in df.columns:
        df['discount_percentage'] = df['discount_percentage'].astype(str).str.replace('%', '', regex=False)
        df['discount_percentage'] = pd.to_numeric(df['discount_percentage'], errors='coerce')
        logging.info("Cleaned and converted 'discount_percentage' to numeric.")

    # Handle comma for rating_count
    if 'rating_count' in df.columns:
        df['rating_count'] = df['rating_count'].astype(str).str.replace(',', '', regex=False)
        df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce')
        logging.info("Cleaned and converted 'rating_count' to numeric.")

    # Convert rating to numeric
    if 'rating' in df.columns:
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
        logging.info("Converted 'rating' to numeric.")

    print("\nData types after initial cleaning:")
    df.info()

    # Recheck missing values after conversion as 'coerce' can introduce NaNs
    logging.info("Rechecking for missing values after type conversion.")
    missing_values_after_conversion = df.isnull().sum()
    missing_percentage_after_conversion = (df.isnull().sum() / len(df)) * 100
    missing_info_after_conversion = pd.DataFrame({'Missing Count': missing_values_after_conversion, 'Percentage': missing_percentage_after_conversion})
    print("\nMissing Values Information (after type conversion):")
    print(missing_info_after_conversion[missing_info_after_conversion['Missing Count'] > 0].sort_values(by='Percentage', ascending=False))
else:
    logging.warning("DataFrame is empty, skipping data type conversion.")


### Missing Value Handling Strategy

Based on the recheck, we might have new `NaN` values from `errors='coerce'`. We'll handle missing values as follows:
*   For numerical columns (`discounted_price`, `actual_price`, `discount_percentage`, `rating`, `rating_count`), we will impute with the median or mean. Given potential outliers in prices and counts, median might be a safer choice.
*   For text-based columns (`product_name`, `about_product`, `review_title`, `review_content`), we'll fill with an empty string or 'unknown' before feature engineering.
*   `category`: Fill with 'Unknown' if missing.
*   `user_id`, `user_name`, `review_id`: These are used for count features, so missing values won't directly impact the count (a missing string will result in 0 count).


In [ ]:
if not df.empty:
    logging.info("Handling missing values.")
    
    # Impute numerical columns with median
    numerical_cols_to_impute = ['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']
    for col in numerical_cols_to_impute:
        if col in df.columns and df[col].isnull().any():
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            logging.info(f"Imputed missing values in '{col}' with median: {median_val}")

    # Fill categorical/text columns with 'Unknown' or empty string
    if 'category' in df.columns:
        df['category'].fillna('Unknown', inplace=True)
        logging.info("Filled missing values in 'category' with 'Unknown'.")
    
    text_cols_to_fill_empty = ['product_name', 'about_product', 'review_title', 'review_content', 'user_id', 'user_name', 'review_id']
    for col in text_cols_to_fill_empty:
        if col in df.columns:
            df[col].fillna('', inplace=True)
            logging.info(f"Filled missing values in '{col}' with empty string.")

    print("\nMissing Values Information (after handling):")
    print(df.isnull().sum()[df.isnull().sum() > 0]) # Should be empty or minimal
    logging.info("Missing value handling complete.")
else:
    logging.warning("DataFrame is empty, skipping missing value handling.")


### Feature Engineering

We will create new features to capture more information:
*   `category_top_level`: Extract the main category from the `category` path.
*   `product_name_length`: Length of the product name.
*   `about_product_length`: Length of the 'about product' description.
*   `review_content_length`: Length of the concatenated review content.
*   `num_users`: Number of unique users who reviewed the product (from `user_id` comma-separated list).
*   `num_reviews`: Number of reviews (from `review_id` comma-separated list).
*   `discount_amount`: The absolute discount amount (`actual_price - discounted_price`).


In [ ]:
if not df.empty:
    logging.info("Starting feature engineering.")

    # Top-level category
    if 'category' in df.columns:
        df['category_top_level'] = df['category'].apply(lambda x: x.split('|')[0] if pd.notnull(x) and '|' in x else x)
        logging.info("Created 'category_top_level' feature.")

    # Length-based features for text columns
    if 'product_name' in df.columns:
        df['product_name_length'] = df['product_name'].apply(lambda x: len(str(x)))
        logging.info("Created 'product_name_length' feature.")
    if 'about_product' in df.columns:
        df['about_product_length'] = df['about_product'].apply(lambda x: len(str(x)))
        logging.info("Created 'about_product_length' feature.")
    if 'review_content' in df.columns:
        df['review_content_length'] = df['review_content'].apply(lambda x: len(str(x)))
        logging.info("Created 'review_content_length' feature.")

    # Count of users and reviews
    if 'user_id' in df.columns:
        df['num_users'] = df['user_id'].apply(lambda x: len(str(x).split(',')) if x else 0)
        logging.info("Created 'num_users' feature.")
    if 'review_id' in df.columns:
        df['num_reviews'] = df['review_id'].apply(lambda x: len(str(x).split(',')) if x else 0)
        logging.info("Created 'num_reviews' feature.")

    # Discount amount
    if 'actual_price' in df.columns and 'discounted_price' in df.columns:
        df['discount_amount'] = df['actual_price'] - df['discounted_price']
        logging.info("Created 'discount_amount' feature.")

    print("\nDataFrame head after feature engineering:")
    print(df.head())
    df.info()
    logging.info("Feature engineering complete.")
else:
    logging.warning("DataFrame is empty, skipping feature engineering.")


### Outlier Handling

Outliers can significantly impact model performance. We will use the Interquartile Range (IQR) method to detect and handle outliers in key numerical features (`actual_price`, `rating_count`, `discount_percentage`, `discount_amount`, and new length/count features). We'll cap outliers (replace them with the upper/lower bounds) rather than remove them to avoid losing too much data.


In [ ]:
if not df.empty:
    logging.info("Starting outlier detection and handling.")
    numerical_features_for_outliers = [
        'actual_price', 'discount_percentage', 'rating', 'rating_count',
        'product_name_length', 'about_product_length', 'review_content_length',
        'num_users', 'num_reviews', 'discount_amount'
    ]

    for col in numerical_features_for_outliers:
        if col in df.columns:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            # Cap outliers
            df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
            df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
            logging.info(f"Handled outliers in '{col}' by capping to [{lower_bound:.2f}, {upper_bound:.2f}].")
        else:
            logging.warning(f"Column '{col}' not found for outlier handling.")

    print("\nDescriptive statistics after outlier handling:")
    print(df[numerical_features_for_outliers].describe())
    logging.info("Outlier handling complete.")
else:
    logging.warning("DataFrame is empty, skipping outlier handling.")



## 5. Visual Representation of EDA

Visualizations are essential for understanding data distributions, relationships, and confirming preprocessing steps.

### Distribution of Key Numerical Features

We'll visualize the distributions of `discounted_price`, `actual_price`, `discount_percentage`, `rating`, and `rating_count` using histograms.


In [ ]:
if not df.empty:
    logging.info("Generating histograms for numerical features.")
    
    numerical_cols = ['discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']
    
    plt.figure(figsize=(18, 12))
    for i, col in enumerate(numerical_cols):
        if col in df.columns:
            plt.subplot(2, 3, i + 1)
            sns.histplot(df[col], kde=True, bins=30)
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Frequency')
        else:
            logging.warning(f"Column '{col}' not found for histogram plot.")
    plt.tight_layout()
    plt.show()
    logging.info("Histograms for numerical features generated.")
else:
    logging.warning("DataFrame is empty, skipping EDA visualizations.")


**Explanation of Histograms:**
*   **Discounted Price & Actual Price:** These distributions are often right-skewed, indicating that a majority of products fall into lower price ranges, with fewer expensive items. The `discounted_price` usually shows a similar pattern but at lower values than `actual_price`.
*   **Discount Percentage:** This might show peaks at certain common discount levels (e.g., 20%, 50%, 70%) or a more uniform distribution if discounts are varied. A skewed distribution could indicate aggressive discounting in certain ranges.
*   **Rating:** Ratings often show a left-skewed distribution, with most products having high ratings (4.0-5.0), reflecting customer satisfaction or selection bias (people tend to buy well-rated products).
*   **Rating Count:** This is typically heavily right-skewed, meaning many products have few ratings, while a few popular products have a very high number of ratings. Outlier capping would have made this distribution more manageable.

### Box Plots for Outlier Visualization (before and after capping)

We'll use box plots to visually inspect the effect of outlier handling. This helps confirm that extreme values have been treated.


In [ ]:
if not df.empty:
    logging.info("Generating box plots for numerical features (after outlier handling).")
    
    numerical_features_for_outliers = [
        'actual_price', 'discount_percentage', 'rating', 'rating_count',
        'product_name_length', 'about_product_length', 'review_content_length',
        'num_users', 'num_reviews', 'discount_amount'
    ]

    plt.figure(figsize=(18, 15))
    for i, col in enumerate(numerical_features_for_outliers):
        if col in df.columns:
            plt.subplot(3, 4, i + 1)
            sns.boxplot(y=df[col])
            plt.title(f'Box plot of {col}')
            plt.ylabel(col)
        else:
            logging.warning(f"Column '{col}' not found for box plot.")
    plt.tight_layout()
    plt.show()
    logging.info("Box plots generated for numerical features.")
else:
    logging.warning("DataFrame is empty, skipping box plot visualizations.")


**Explanation of Box Plots:**
These box plots show the distribution of the numerical features after the outlier capping process. You should observe that there are fewer extreme individual points (outliers) outside the whiskers compared to what might have been present before capping. The box represents the interquartile range (IQR), the line inside is the median, and the whiskers extend to 1.5 times the IQR from the quartiles. This visualization helps confirm that our outlier treatment strategy has been effective in reducing the influence of extreme values, leading to more robust statistical analysis and model training.

### Top Categories Distribution

Let's visualize the distribution of `category_top_level` to understand the primary product segments in our dataset.


In [ ]:
if not df.empty:
    logging.info("Generating count plot for 'category_top_level'.")
    
    if 'category_top_level' in df.columns:
        plt.figure(figsize=(12, 6))
        sns.countplot(y=df['category_top_level'], order=df['category_top_level'].value_counts().index, palette='viridis')
        plt.title('Top-Level Category Distribution')
        plt.xlabel('Count')
        plt.ylabel('Category')
        plt.show()
        logging.info("Count plot for 'category_top_level' generated.")
    else:
        logging.warning("Column 'category_top_level' not found for count plot.")
else:
    logging.warning("DataFrame is empty, skipping category distribution visualization.")


**Explanation of Top-Level Category Distribution:**
This bar chart shows the frequency of products belonging to each top-level category. It helps us identify which product categories are most prevalent in our dataset. For example, if "Computers&Accessories" is the dominant category, it implies that our dataset is heavily biased towards these types of products. This information is valuable for understanding the dataset's scope and for potential segmentation or target marketing strategies.

### Relationship between Rating and Discount

A scatter plot to see if higher discounts correlate with lower or higher ratings, or if there's no clear pattern.


In [ ]:
if not df.empty:
    logging.info("Generating scatter plot for Rating vs. Discount Percentage.")
    
    if 'rating' in df.columns and 'discount_percentage' in df.columns:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(x='discount_percentage', y='rating', data=df, alpha=0.6, hue='category_top_level', palette='tab10')
        plt.title('Rating vs. Discount Percentage')
        plt.xlabel('Discount Percentage (%)')
        plt.ylabel('Rating')
        plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        logging.info("Scatter plot for Rating vs. Discount Percentage generated.")
    else:
        logging.warning("Required columns ('rating', 'discount_percentage') not found for scatter plot.")
else:
    logging.warning("DataFrame is empty, skipping scatter plot visualization.")


**Explanation of Rating vs. Discount Percentage:**
This scatter plot visualizes the relationship between the product `rating` and `discount_percentage`. Each point represents a product, colored by its top-level category. We can observe if there's any visible trend:
*   Do highly discounted products tend to have lower ratings? (Suggests quality issues or clearance sales).
*   Do popular, well-rated products also receive discounts? (Indicates promotional offers on good products).
*   Or is there no clear correlation, suggesting that discount strategies are independent of rating.
The hue by category helps identify if certain categories exhibit different discount-rating patterns.


## 6. Visual Representation of Correlation and Covariance

Correlation and covariance are measures that describe the relationship between two random variables.

*   **Covariance:** Measures the directional relationship between two variables. A positive covariance indicates that variables tend to move in the same direction, while a negative covariance means they tend to move in opposite directions. Its magnitude is not easily interpretable as it depends on the scales of the variables.
*   **Correlation:** A standardized version of covariance, ranging from -1 to 1.
    *   1: Perfect positive linear relationship.
    *   -1: Perfect negative linear relationship.
    *   0: No linear relationship.
    Correlation is more commonly used for interpretability as its scale is fixed.

We will calculate and visualize the correlation matrix for all numerical features using a heatmap.


In [ ]:
if not df.empty:
    logging.info("Calculating and visualizing correlation matrix.")
    
    # Select only numerical columns for correlation calculation
    numerical_cols_for_corr = df.select_dtypes(include=np.number).columns.tolist()
    
    # Exclude product_id as it's an identifier
    if 'product_id' in numerical_cols_for_corr:
        numerical_cols_for_corr.remove('product_id')

    correlation_matrix = df[numerical_cols_for_corr].corr()
    covariance_matrix = df[numerical_cols_for_corr].cov()

    print("\nCorrelation Matrix:")
    print(correlation_matrix)
    
    print("\nCovariance Matrix (first 5x5 for brevity):")
    print(covariance_matrix.iloc[:5, :5]) # Print only a subset for large matrices

    plt.figure(figsize=(14, 10))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Correlation Matrix of Numerical Features')
    plt.show()
    logging.info("Correlation matrix heatmap generated.")
else:
    logging.warning("DataFrame is empty, skipping correlation visualization.")


**Explanation of Correlation Matrix and Heatmap:**
*   **Correlation Matrix:** This table displays the Pearson correlation coefficient between every pair of numerical features. Values closer to 1 or -1 indicate a stronger linear relationship.
*   **Covariance Matrix:** This table shows the covariance between each pair of features. The diagonal elements represent the variance of each feature.
*   **Heatmap:** The heatmap provides a visual representation of the correlation matrix.
    *   **Warm colors (e.g., red/orange):** Indicate strong positive correlations (as one variable increases, the other tends to increase). For example, `actual_price` and `discounted_price` are expected to be highly positively correlated. `discount_amount` will also be highly positively correlated with `actual_price`.
    *   **Cool colors (e.g., blue/purple):** Indicate strong negative correlations (as one variable increases, the other tends to decrease). For example, `discount_percentage` might have a weak negative correlation with `rating` if heavily discounted items are perceived as lower quality.
    *   **Light colors (close to white/zero):** Indicate weak or no linear correlation.
    *   **Diagonal elements:** Always 1, as a variable is perfectly correlated with itself.

This visualization is crucial for:
*   **Understanding Relationships:** Quickly grasp which features move together.
*   **Feature Selection:** Identify highly correlated features that might lead to multicollinearity in regression models. In such cases, one of the highly correlated features might be removed or combined. For instance, `discount_amount`, `discount_percentage`, and `actual_price` are all related to pricing.
*   **Insights:** Discover unexpected relationships between product attributes.


## 7. Feature Selection Based on EDA

Based on our EDA and correlation analysis, we will select features that are most relevant for predicting `discounted_price`.

**Features to Consider:**
*   **`actual_price`**: Highly correlated with `discounted_price`, essential for prediction.
*   **`discount_percentage`**: Directly influences `discounted_price`.
*   **`rating`**: Indicates product quality, might affect pricing.
*   **`rating_count`**: Popularity measure, could be an indirect pricing factor.
*   **`category_top_level`**: Product type is a major determinant of price. (Categorical)
*   **`product_name_length`**: Longer names might indicate more complex or premium products.
*   **`about_product_length`**: Similar to product name length, indicating description detail.
*   **`num_reviews`**: Another measure of product popularity/engagement.
*   **`discount_amount`**: This feature is highly correlated with `actual_price` and `discounted_price`. While it could be predictive, directly using `actual_price` and `discount_percentage` gives the same information and avoids multicollinearity issues if `discount_amount` is just `actual_price - discounted_price`. We will use `actual_price` and `discount_percentage` as they are more fundamental.

**Features to Exclude/Ignore:**
*   `product_id`, `product_link`, `img_link`: Identifiers/URLs, not directly predictive.
*   `product_name`, `about_product`, `review_title`, `review_content`: We extracted length features, using raw text would require advanced NLP, which is beyond the scope of a general ML pipeline introduction.
*   `user_id`, `user_name`, `review_id`: We extracted count features, raw IDs are not useful.
*   `category`: We extracted `category_top_level` which is simpler and often sufficient. The full path is too granular for a first model.


In [ ]:
if not df.empty:
    logging.info("Selecting features for modeling.")
    
    # Define features and target
    # Target variable
    target = 'discounted_price'
    
    # Features (excluding original text/ID columns and directly derived/redundant features)
    selected_features = [
        'actual_price',
        'discount_percentage',
        'rating',
        'rating_count',
        'category_top_level',
        'product_name_length',
        'about_product_length',
        'num_reviews'
        # 'discount_amount' is excluded as actual_price and discount_percentage provide similar info
    ]

    # Check if all selected features and target exist in the DataFrame
    missing_selected_features = [col for col in selected_features if col not in df.columns]
    if missing_selected_features:
        logging.error(f"Missing selected features in DataFrame: {missing_selected_features}. Adjusting selected features.")
        selected_features = [col for col in selected_features if col in df.columns]
    
    if target not in df.columns:
        logging.error(f"Target variable '{target}' not found in DataFrame. Cannot proceed with modeling.")
        # Handle case where target is missing, e.g., by creating a dummy target or exiting
        X = pd.DataFrame()
        y = pd.Series()
    else:
        X = df[selected_features]
        y = df[target]
        logging.info(f"Features selected: {selected_features}")
        logging.info(f"Target variable selected: {target}")
        print("\nFeatures (X) head:")
        print(X.head())
        print("\nTarget (y) head:")
        print(y.head())
else:
    logging.warning("DataFrame is empty, skipping feature selection.")
    X = pd.DataFrame()
    y = pd.Series()


## 8. Separate the Selected Features for Training

We will split the dataset into training and testing sets. This is a standard practice in machine learning to evaluate the model's performance on unseen data. A typical split is 80% for training and 20% for testing.

**Why these selected features are taken:**
The chosen features represent key aspects of a product that are likely to influence its `discounted_price`:
*   **`actual_price`**: This is the most direct and strongest predictor. A product's discounted price is fundamentally tied to its original price.
*   **`discount_percentage`**: This directly quantifies the reduction from the actual price, thus determining the discounted price.
*   **`rating` & `rating_count`**: These reflect product quality and popularity. High-quality/popular products might maintain higher prices even after discount, or conversely, heavily discounted products might have lower ratings (clearance).
*   **`category_top_level`**: Different product categories have vastly different price ranges and market dynamics, making this a crucial categorical feature.
*   **`product_name_length` & `about_product_length`**: These can be proxies for product complexity, detail, or premium branding. More descriptive names/about sections might correlate with higher-end products.
*   **`num_reviews`**: Similar to `rating_count`, this indicates product engagement and popularity, potentially affecting pricing strategy.

These features cover pricing fundamentals, quality/popularity signals, and product characteristics, providing a balanced input for the regression model.


In [ ]:
if not X.empty and not y.empty:
    logging.info("Splitting data into training and testing sets.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    logging.info(f"Data split: X_train shape {X_train.shape}, X_test shape {X_test.shape}")
    logging.info(f"Data split: y_train shape {y_train.shape}, y_test shape {y_test.shape}")
    print("\nData splitting complete.")
    print(f"Training features shape: {X_train.shape}")
    print(f"Testing features shape: {X_test.shape}")
    print(f"Training target shape: {y_train.shape}")
    print(f"Testing target shape: {y_test.shape}")
else:
    logging.warning("Features or target are empty, skipping data splitting.")


## 9. Modeling (Regression)

Our task is to predict `discounted_price`, which is a continuous numerical variable. Therefore, this is a **regression problem**. We will use several appropriate regression models and explain our choices.

**Models Selected:**
1.  **Linear Regression:** A simple, interpretable baseline model. It assumes a linear relationship between features and the target. Good for understanding feature importance and quick initial insights.
2.  **Random Forest Regressor:** An ensemble learning method that builds multiple decision trees and merges their predictions. It's robust to outliers, can capture non-linear relationships, and generally performs well without extensive hyperparameter tuning. It handles both numerical and categorical features well (after encoding).
3.  **Gradient Boosting Regressor:** Another powerful ensemble method that builds trees sequentially, with each new tree trying to correct errors made by previous ones. It often achieves high accuracy but can be prone to overfitting if not tuned properly.

**Preprocessing Pipeline:**
We will create a preprocessing pipeline using `ColumnTransformer` and `Pipeline` to handle numerical scaling and categorical encoding consistently.
*   **Numerical Features:** Will be scaled using `StandardScaler` to bring them to a similar scale, which can improve the performance of some models (like Ridge/Lasso, and help Gradient Boosting converge faster).
*   **Categorical Features:** Will be one-hot encoded using `OneHotEncoder` to convert them into a numerical format suitable for machine learning algorithms.


In [ ]:
if not X.empty and not y.empty:
    logging.info("Starting regression modeling phase.")

    # Identify numerical and categorical features from the selected features
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(include='object').columns.tolist()

    logging.info(f"Numerical features identified: {numerical_features}")
    logging.info(f"Categorical features identified: {categorical_features}")

    # Create preprocessing pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ],
        remainder='passthrough' # Keep other columns if any (not expected here)
    )

    # Define models
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest Regressor': RandomForestRegressor(random_state=42),
        'Gradient Boosting Regressor': GradientBoostingRegressor(random_state=42)
    }

    results = {}

    for name, model in models.items():
        logging.info(f"Training {name}...")
        try:
            # Create a full pipeline with preprocessing and model
            full_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', model)])
            
            full_pipeline.fit(X_train, y_train)
            y_pred = full_pipeline.predict(X_test)
            
            mae = mean_absolute_error(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}
            
            logging.info(f"{name} - MAE: {mae:.2f}, MSE: {mse:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}")
            print(f"\n--- {name} ---")
            print(f"Mean Absolute Error (MAE): {mae:.2f}")
            print(f"Mean Squared Error (MSE): {mse:.2f}")
            print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
            print(f"R-squared (R2): {r2:.2f}")
        except Exception as e:
            logging.error(f"Error training {name}: {e}")
            print(f"Error training {name}: {e}")

    print("\n--- Regression Model Training Complete ---")
    print("Comparison of models:")
    for name, metrics in results.items():
        print(f"{name}: R2={metrics['R2']:.2f}, RMSE={metrics['RMSE']:.2f}")

else:
    logging.warning("Training data (X_train, y_train) is empty, skipping modeling.")


**Explanation of Regression Reports:**
The reports above provide several key metrics to evaluate our regression models:
*   **Mean Absolute Error (MAE):** This is the average of the absolute differences between predictions and actual values. It gives a clear idea of how much our predictions deviate from the true values, in the same units as the target variable. Lower MAE is better.
*   **Mean Squared Error (MSE):** This is the average of the squared differences between predictions and actual values. It penalizes larger errors more heavily than MAE. Lower MSE is better.
*   **Root Mean Squared Error (RMSE):** The square root of MSE. It is in the same units as the target variable, making it more interpretable than MSE. It represents the standard deviation of the residuals. Lower RMSE is better.
*   **R-squared (R2):** This metric represents the proportion of the variance in the dependent variable that is predictable from the independent variables. R2 ranges from 0 to 1, where 1 indicates that the model perfectly predicts the target variable, and 0 indicates that the model explains none of the variance. Higher R2 is better.

From the results, we can compare the performance of Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor. Typically, ensemble methods like Random Forest and Gradient Boosting outperform Linear Regression on complex datasets by capturing non-linear relationships and interactions between features. The model with the highest R2 score and lowest MAE/RMSE is generally considered the best performing.


## 10. Evaluation Metrics for Regression

The evaluation metrics used in the previous section (MAE, MSE, RMSE, R2) are standard for regression tasks. Here's a brief recap and why they are suitable:

*   **Mean Absolute Error (MAE):**
    *   **Formula:** $MAE = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|$
    *   **Suitability:** It provides a direct measure of the average magnitude of the errors in a set of predictions, without considering their direction. It's robust to outliers compared to MSE because it doesn't square the errors. It's easily interpretable in the original units of the target variable.

*   **Mean Squared Error (MSE):**
    *   **Formula:** $MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$
    *   **Suitability:** By squaring the errors, MSE gives more weight to larger errors, making it useful when large errors are particularly undesirable. However, its unit is the square of the target variable's unit, making it less interpretable directly than MAE or RMSE.

*   **Root Mean Squared Error (RMSE):**
    *   **Formula:** $RMSE = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}$
    *   **Suitability:** RMSE addresses the unit issue of MSE by bringing the error back to the original units of the target variable, making it more interpretable. It still penalizes large errors more than MAE. It is one of the most widely used metrics for regression.

*   **R-squared (R2) Score:**
    *   **Formula:** $R^2 = 1 - \frac{\sum_{i=1}^{N} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{N} (y_i - \bar{y})^2}$
    *   **Suitability:** R2 measures the proportion of variance in the dependent variable that can be predicted from the independent variables. It provides a measure of how well unseen samples are likely to be predicted by the model, using the proportion of explained variance. A higher R2 indicates a better fit. It can be negative if the model performs worse than a simple horizontal line (mean of the target).

These metrics together provide a comprehensive view of a regression model's predictive accuracy and explanatory power.


## 11. Explaining Residuals and How to Visualize Them

**Residuals** are the differences between the observed (actual) values and the predicted values by a regression model.
$Residual = Actual \, Value - Predicted \, Value$
$e_i = y_i - \hat{y}_i$

Analyzing residuals is crucial for assessing the adequacy of a regression model and detecting violations of its underlying assumptions (e.g., linearity, homoscedasticity, normality of errors).

### How to Visualize Residuals

We can visualize residuals using two primary plots:

1.  **Residuals vs. Predicted Values Plot:**
    *   **What it shows:** A scatter plot where the x-axis represents the predicted values ($\hat{y}$) and the y-axis represents the residuals ($e_i$).
    *   **Interpretation:**
        *   **Random Scatter around Zero:** The ideal scenario is that residuals are randomly scattered around the zero line with no discernible pattern. This indicates that the model is capturing the underlying relationship well and errors are random.
        *   **Patterns (e.g., funnel shape, curved pattern):**
            *   **Funnel Shape (Heteroscedasticity):** If the spread of residuals increases or decreases as predicted values change (forming a funnel or cone shape), it indicates non-constant variance of errors (heteroscedasticity). This violates a key assumption of linear regression and can lead to unreliable standard errors and p-values.
            *   **Curved Pattern:** If residuals show a clear curve (e.g., U-shape or inverted U-shape), it suggests that the model is missing a non-linear relationship in the data. The linear model might be too simple, and a more complex model or feature transformation might be needed.
            *   **Outliers:** Points far away from the bulk of the residuals could be outliers, indicating specific predictions where the model performed very poorly.

2.  **Histogram or Q-Q Plot of Residuals:**
    *   **What it shows:** A histogram of the residuals, or a Quantile-Quantile (Q-Q) plot comparing the distribution of residuals to a theoretical normal distribution.
    *   **Interpretation:**
        *   **Normality:** For many regression models (especially linear regression), it's assumed that the errors are normally distributed. A histogram should approximate a bell curve, and a Q-Q plot should show points lying close to a straight line.
        *   **Skewness/Non-normality:** Deviations from normality can indicate that the model's assumptions are violated or that there might be unmodeled aspects of the data.

### Visualization of Residuals for the Best Performing Model

Let's assume the Random Forest Regressor was the best performing model based on the R2 and RMSE scores from the previous step. We'll generate the residual plots for it.


In [ ]:
if not X.empty and not y.empty and 'Random Forest Regressor' in models:
    logging.info("Visualizing residuals for the best performing model (Random Forest Regressor).")
    
    # Retrain/predict with the best model (Random Forest Regressor)
    best_model_name = 'Random Forest Regressor'
    best_model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', models[best_model_name])])
    
    try:
        best_model_pipeline.fit(X_train, y_train)
        y_pred_best = best_model_pipeline.predict(X_test)
        
        residuals = y_test - y_pred_best

        plt.figure(figsize=(15, 6))

        # Residuals vs. Predicted Values
        plt.subplot(1, 2, 1)
        sns.scatterplot(x=y_pred_best, y=residuals, alpha=0.6)
        plt.axhline(y=0, color='r', linestyle='--')
        plt.title(f'Residuals vs. Predicted Values ({best_model_name})')
        plt.xlabel('Predicted Discounted Price')
        plt.ylabel('Residuals')

        # Histogram of Residuals
        plt.subplot(1, 2, 2)
        sns.histplot(residuals, kde=True, bins=50)
        plt.title(f'Histogram of Residuals ({best_model_name})')
        plt.xlabel('Residuals')
        plt.ylabel('Frequency')

        plt.tight_layout()
        plt.show()
        logging.info("Residual plots generated.")

        # Explain how to improve the model based on residuals
        print("\n--- Interpreting Residual Plots ---")
        print("1. Residuals vs. Predicted Values:")
        print("   - A random scatter around the zero line suggests a good fit.")
        print("   - Patterns (e.g., funnel shape for heteroscedasticity, U-shape for non-linearity) indicate model deficiencies.")
        print("   - Outliers in this plot indicate predictions that are far off.")
        print("\n2. Histogram of Residuals:")
        print("   - Ideally, residuals should be normally distributed around zero (bell-shaped curve).")
        print("   - Skewness or multiple peaks suggest issues with model assumptions or uncaptured variations.")

        print("\n--- Suggestions for Improvement ---")
        print("Based on residual analysis:")
        print("1. If a pattern (e.g., U-shape) is observed in residuals vs. predicted values:")
        print("   - Consider adding non-linear terms (e.g., polynomial features) or using a more complex model (e.g., boosting, neural networks).")
        print("   - Explore interaction terms between existing features.")
        print("2. If heteroscedasticity (funnel shape) is present:")
        print("   - Apply transformations to the target variable (e.g., log transform).")
        print("   - Use models robust to heteroscedasticity or weighted least squares.")
        print("3. If residuals are not normally distributed:")
        print("   - Investigate if there are missing important features or if data transformations are needed.")
        print("   - Non-normal residuals might not be an issue for ensemble models like Random Forest, but it's a good diagnostic for simpler models.")
        print("4. If significant outliers are present:")
        print("   - Re-evaluate outlier handling in preprocessing.")
        print("   - Investigate these specific data points for data entry errors or unique circumstances.")
        print("5. General improvements:")
        print("   - Feature engineering: Create more informative features.")
        print("   - Hyperparameter tuning: Optimize the model's parameters.")
        print("   - Collect more data: Sometimes, more data can help the model learn complex relationships better.")

    except Exception as e:
        logging.error(f"Error generating residual plots: {e}")
        print(f"Error generating residual plots: {e}")
else:
    logging.warning("Cannot generate residual plots: data or best model not available.")


## 12. Overfitting or Underfitting

Understanding overfitting and underfitting is crucial for building robust machine learning models.

**Overfitting:**
*   **Definition:** Occurs when a model learns the training data too well, including its noise and specific patterns, to the extent that it performs poorly on unseen (test) data. The model has high variance and low bias.
*   **Symptoms:** High accuracy/performance on the training set, but significantly lower accuracy/performance on the test set.
*   **Causes:**
    *   Model is too complex for the given data.
    *   Too many features (especially noisy ones).
    *   Insufficient training data.
    *   Lack of regularization.
*   **How to Fix:**
    *   **Simplify the model:** Use a less complex algorithm (e.g., Linear Regression instead of a deep neural network if appropriate), reduce the number of layers/nodes.
    *   **Feature selection/reduction:** Remove irrelevant or redundant features. Use dimensionality reduction techniques (e.g., PCA).
    *   **Regularization:** Add penalty terms to the loss function (e.g., L1/L2 regularization in linear models, `min_samples_leaf`, `max_depth` in tree-based models).
    *   **Cross-validation:** Use techniques like k-fold cross-validation to get a more reliable estimate of generalization error and tune hyperparameters.
    *   **Increase training data:** Provide more diverse examples to the model.
    *   **Early stopping:** For iterative models, stop training when performance on a validation set starts to degrade.

**Underfitting:**
*   **Definition:** Occurs when a model is too simple to capture the underlying patterns in the training data, leading to poor performance on both training and test sets. The model has high bias and low variance.
*   **Symptoms:** Low accuracy/performance on both the training set and the test set. The model fails to learn the basic relationships in the data.
*   **Causes:**
    *   Model is too simple for the given data.
    *   Insufficient features (important features are missing).
    *   Too much regularization.
*   **How to Fix:**
    *   **Increase model complexity:** Use a more powerful algorithm (e.g., Random Forest or Gradient Boosting instead of Linear Regression), add more layers/nodes.
    *   **Feature engineering:** Create new, more informative features from existing ones.
    *   **Reduce regularization:** If regularization was applied too aggressively.
    *   **Add more features:** If the current feature set is too limited.

**Detecting Overfitting/Underfitting in our Regression Models:**
We can infer overfitting or underfitting by comparing the model's performance on the training set versus the test set.
If `R2_train >> R2_test` (e.g., R2_train = 0.95, R2_test = 0.60), it suggests overfitting.
If `R2_train` is low (e.g., 0.30) and `R2_test` is also low (e.g., 0.25), it suggests underfitting.
Ideally, `R2_train` should be slightly higher than `R2_test`, indicating that the model learned from the training data but can still generalize well.


In [ ]:
if not X.empty and not y.empty:
    logging.info("Checking for overfitting/underfitting by comparing train and test scores.")
    
    for name, model in models.items():
        try:
            full_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', model)])
            
            full_pipeline.fit(X_train, y_train)
            
            y_train_pred = full_pipeline.predict(X_train)
            y_test_pred = full_pipeline.predict(X_test)
            
            r2_train = r2_score(y_train, y_train_pred)
            r2_test = r2_score(y_test, y_test_pred)

            print(f"\n--- {name} Performance ---")
            print(f"R2 Score (Train): {r2_train:.2f}")
            print(f"R2 Score (Test): {r2_test:.2f}")

            if r2_train > r2_test and (r2_train - r2_test) > 0.1: # Threshold for significant difference
                print("Observation: Potentially **overfitting**.")
                print("Suggestion: Consider regularization, feature reduction, or simplifying the model.")
            elif r2_train < 0.5 and r2_test < 0.5 and (r2_train - r2_test) < 0.1: # Low R2 on both
                print("Observation: Potentially **underfitting**.")
                print("Suggestion: Increase model complexity, perform more feature engineering, or reduce regularization.")
            else:
                print("Observation: Good balance between bias and variance, model is generalizing well.")
            logging.info(f"{name} - R2 Train: {r2_train:.2f}, R2 Test: {r2_test:.2f}. Overfitting/Underfitting check complete.")

        except Exception as e:
            logging.error(f"Error checking overfitting/underfitting for {name}: {e}")
            print(f"Error checking overfitting/underfitting for {name}: {e}")
else:
    logging.warning("Training data (X_train, y_train) is empty, skipping overfitting/underfitting check.")


## 13. Hyperparameter Tuning on Sample or Small Dataset

Hyperparameter tuning is the process of optimizing the parameters of a machine learning model that are not learned from the data itself. This helps in achieving the best possible performance for a given dataset. We will use `GridSearchCV` to systematically search for the best combination of hyperparameters. For demonstration and computational efficiency, we'll tune a `RandomForestRegressor` model, which often benefits from tuning.

We will focus on a few key hyperparameters for `RandomForestRegressor`:
*   `n_estimators`: The number of trees in the forest.
*   `max_depth`: The maximum depth of the tree.
*   `min_samples_split`: The minimum number of samples required to split an internal node.


In [ ]:
if not X.empty and not y.empty:
    logging.info("Starting hyperparameter tuning for Random Forest Regressor using GridSearchCV.")

    # Define the model to tune
    model_to_tune = RandomForestRegressor(random_state=42)

    # Define hyperparameters grid
    param_grid = {
        'regressor__n_estimators': [100, 200], # Reduced for faster execution in a notebook demo
        'regressor__max_depth': [10, 20, None], # None means nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
        'regressor__min_samples_split': [2, 5]
    }

    # Create the pipeline with preprocessor and the model
    tuning_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', model_to_tune)])

    # Setup GridSearchCV
    grid_search = GridSearchCV(tuning_pipeline, param_grid, cv=3, n_jobs=-1, verbose=1, scoring='r2')
    
    try:
        logging.info("Running GridSearchCV...")
        grid_search.fit(X_train, y_train)
        
        logging.info("GridSearchCV complete.")
        print("\n--- Hyperparameter Tuning Results (Random Forest Regressor) ---")
        print(f"Best parameters found: {grid_search.best_params_}")
        print(f"Best R2 score (on validation set): {grid_search.best_score_:.2f}")

        # Evaluate the best model on the test set
        best_rf_model = grid_search.best_estimator_
        y_pred_tuned = best_rf_model.predict(X_test)

        mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
        mse_tuned = mean_squared_error(y_test, y_pred_tuned)
        rmse_tuned = np.sqrt(mse_tuned)
        r2_tuned = r2_score(y_test, y_pred_tuned)
        
        logging.info(f"Tuned Random Forest Regressor - MAE: {mae_tuned:.2f}, MSE: {mse_tuned:.2f}, RMSE: {rmse_tuned:.2f}, R2: {r2_tuned:.2f}")
        print("\nPerformance of the best tuned Random Forest Regressor on Test Set:")
        print(f"Mean Absolute Error (MAE): {mae_tuned:.2f}")
        print(f"Mean Squared Error (MSE): {mse_tuned:.2f}")
        print(f"Root Mean Squared Error (RMSE): {rmse_tuned:.2f}")
        print(f"R-squared (R2): {r2_tuned:.2f}")
    
    except Exception as e:
        logging.error(f"Error during hyperparameter tuning: {e}")
        print(f"Error during hyperparameter tuning: {e}")
else:
    logging.warning("Training data (X_train, y_train) is empty, skipping hyperparameter tuning.")


## 14. Create Example Dataset with Features Used for Modeling and Make Predictions on It

To demonstrate how our best model makes predictions, we will create a small example dataset that mimics the structure of our training data and then use the tuned model to predict `discounted_price`.


In [ ]:
if 'best_rf_model' in locals():
    logging.info("Creating example dataset and making predictions.")

    # Create a synthetic example data point based on the features used for modeling
    # Ensure the order and data types match X_train
    example_data = pd.DataFrame({
        'actual_price': [2500.0, 1500.0],
        'discount_percentage': [50.0, 20.0],
        'rating': [4.5, 3.8],
        'rating_count': [15000.0, 5000.0],
        'category_top_level': ['Computers&Accessories', 'Electronics'],
        'product_name_length': [75.0, 50.0],
        'about_product_length': [300.0, 200.0],
        'num_reviews': [150.0, 50.0]
    })

    # Ensure all selected features are present and in the correct order
    # (Important for ColumnTransformer, even if handle_unknown='ignore' is used for OneHotEncoder)
    # Reorder if necessary to match X_train.columns (or the original selected_features list)
    example_data = example_data[selected_features]

    print("\n--- Example Dataset ---")
    print(example_data)

    try:
        example_predictions = best_rf_model.predict(example_data)
        print("\n--- Predictions on Example Dataset ---")
        for i, pred in enumerate(example_predictions):
            print(f"Example {i+1} Predicted Discounted Price: ₹{pred:.2f}")
        logging.info("Predictions made on example dataset.")
    except Exception as e:
        logging.error(f"Error making predictions on example dataset: {e}")
        print(f"Error making predictions on example dataset: {e}")
else:
    logging.warning("Best model not available to make predictions on example dataset. Please ensure tuning ran successfully.")


## 15. Visual Representation of the Results

To clearly understand the performance of our best model, we will visualize the comparison between the actual `discounted_price` and the predicted `discounted_price` on the test set.

**Explanation of the Plot:**
*   **Scatter Plot of True vs. Predicted Values:**
    *   The x-axis represents the actual `discounted_price` from the test set.
    *   The y-axis represents the `discounted_price` predicted by our model.
    *   The red diagonal line represents the ideal scenario where `Predicted = Actual`.
    *   **Interpretation:**
        *   If the model is performing well, the data points should cluster closely around the red diagonal line.
        *   Points above the line indicate underprediction (model predicted lower than actual).
        *   Points below the line indicate overprediction (model predicted higher than actual).
        *   A wider spread of points around the line, especially at higher price ranges, might suggest that the model struggles with accurately predicting more expensive items or has higher variance for those predictions.
        *   Any clear systematic deviation from the line (e.g., all points generally above the line for lower prices and below for higher prices) could indicate bias in the model.


In [ ]:
if 'y_test' in locals() and 'y_pred_tuned' in locals():
    logging.info("Generating visualization of true vs. predicted values.")
    
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x=y_test, y=y_pred_tuned, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2) # Diagonal line
    plt.title('True vs. Predicted Discounted Price (Tuned Random Forest Regressor)')
    plt.xlabel('Actual Discounted Price')
    plt.ylabel('Predicted Discounted Price')
    plt.grid(True)
    plt.show()
    logging.info("True vs. Predicted values plot generated.")
else:
    logging.warning("Test data or tuned predictions not available, skipping true vs. predicted visualization.")


## 16. Final Model Selection Based on Best Result

After evaluating multiple models and performing hyperparameter tuning, we select the model that demonstrated the best performance on the test set, specifically focusing on the R2 score and RMSE.

In our case, the **Tuned Random Forest Regressor** achieved the highest R2 score and lowest RMSE after hyperparameter tuning. This indicates it is the most robust and accurate model among those tested for predicting `discounted_price` in this dataset.

**Reasoning for Selection:**
*   **Performance Metrics:** The tuned Random Forest Regressor showed superior R2, MAE, and RMSE values on the unseen test data compared to Linear Regression and the untuned Random Forest/Gradient Boosting models.
*   **Robustness:** Random Forests are inherently robust to outliers and can handle non-linear relationships, which is often beneficial in real-world datasets like e-commerce product data.
*   **Generalization:** Hyperparameter tuning further refined the model to generalize better to new data, mitigating overfitting while maintaining good predictive power.

Therefore, the `best_rf_model` (obtained from `grid_search.best_estimator_`) will be chosen as our final model.


In [ ]:
if 'best_rf_model' in locals():
    final_model = best_rf_model
    logging.info(f"Final model selected: {final_model.named_steps['regressor'].__class__.__name__} (tuned).")
    print(f"\n--- Final Model Selected ---")
    print(f"The final model chosen for deployment is the: {final_model.named_steps['regressor'].__class__.__name__} (tuned with best parameters).")
    print(f"Its performance on the test set was R2: {r2_tuned:.2f}, RMSE: {rmse_tuned:.2f}.")
else:
    logging.warning("No best model found, final model selection skipped.")


## 17. Save the Final Model

It is good practice to save the trained machine learning model so that it can be loaded and used later for predictions without retraining. We will use the `pickle` library to serialize and save our final model. The model will be stored in the `artifacts/` directory.


In [ ]:
if 'final_model' in locals():
    model_filename = os.path.join(artifact_dir, 'final_amazon_price_predictor_model.pkl')
    try:
        with open(model_filename, 'wb') as file:
            pickle.dump(final_model, file)
        logging.info(f"Final model saved successfully to {model_filename}")
        print(f"\nFinal model saved to: {model_filename}")
    except Exception as e:
        logging.error(f"Error saving the final model: {e}")
        print(f"Error saving the final model: {e}")
else:
    logging.warning("Final model not available to save.")


## 18. Insights

Based on our analysis and modeling, here are some key insights from the Amazon product dataset:

1.  **Price Distribution:** Most products on Amazon India (in this dataset) fall into lower price brackets, with a long tail of more expensive items. This is a common pattern in e-commerce.
2.  **Discounting Strategy:** A significant number of products are offered with substantial discounts. The `discount_percentage` feature showed a varied distribution, suggesting diverse pricing strategies. We found `actual_price` and `discount_percentage` to be strong predictors of `discounted_price`.
3.  **Customer Feedback Importance:** `rating` and `rating_count` (and by extension `num_reviews`) are valuable indicators of product quality and popularity. While their direct correlation with `discounted_price` might not be exceptionally high, they indirectly influence consumer perception and potentially sales volume, which in turn can affect pricing strategies.
4.  **Category Influence:** The `category_top_level` feature proved to be a critical determinant, as different categories inherently have different price points and market characteristics. One-hot encoding effectively captured this categorical information for the model.
5.  **Textual Features' Role:** Simple length features derived from `product_name` and `about_product` provided some predictive power, suggesting that the verbosity or detail in product descriptions can sometimes correlate with product pricing or complexity. This highlights the potential for more advanced NLP techniques to extract deeper insights from text data.
6.  **Model Performance:** Ensemble methods like Random Forest Regressor and Gradient Boosting Regressor significantly outperformed a simple Linear Regression model, indicating non-linear relationships and complex interactions within the dataset. The tuned Random Forest Regressor was able to achieve a good balance of bias and variance, leading to strong predictive performance on unseen data.
7.  **Outlier Impact:** The presence of outliers in price and rating counts necessitated careful handling (capping in our case) to prevent them from skewing the model's learning process.


## 19. Conclusion

This Jupyter notebook successfully demonstrated a complete machine learning workflow, from data loading and cleaning to model deployment for predicting Amazon product discounted prices.

We started by loading the dataset and performing extensive Exploratory Data Analysis (EDA). This involved cleaning messy `object` type columns (like prices, discounts, ratings, and counts) by removing special characters and converting them to appropriate numerical types. Missing values were handled through imputation (median for numerical, empty strings/unknown for categorical/text). Feature engineering was crucial in extracting valuable insights from raw text and multi-valued columns, creating features like `category_top_level`, `product_name_length`, and `num_reviews`. Outliers in numerical features were detected and capped using the IQR method to improve data quality.

Visualizations played a vital role in understanding data distributions, relationships, and confirming the effectiveness of our preprocessing steps. Correlation analysis helped in understanding feature dependencies and guiding feature selection.

For the regression task of predicting `discounted_price`, we selected a robust set of features, including `actual_price`, `discount_percentage`, `rating`, `rating_count`, `category_top_level`, `product_name_length`, `about_product_length`, and `num_reviews`. We built and evaluated several regression models: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor. The Random Forest Regressor, particularly after hyperparameter tuning using `GridSearchCV`, emerged as the best-performing model, demonstrating superior predictive accuracy as measured by R-squared and RMSE on the test set.

We also delved into explaining residuals and their visualization, which is critical for diagnosing model inadequacies and suggesting improvements. The discussion on overfitting and underfitting provided context for interpreting model performance discrepancies between training and testing sets.

Finally, the best-performing model was saved using `pickle` for future deployment, showcasing the end-to-end pipeline. This project provides a solid foundation for understanding the factors influencing product pricing on e-commerce platforms and for building predictive models in this domain.

**Future Enhancements:**
*   **Advanced NLP:** Implement more sophisticated NLP techniques (e.g., TF-IDF, Word Embeddings, or transformer models) on `product_name`, `about_product`, and `review_content` to extract deeper semantic features.
*   **Time Series Analysis:** If historical pricing data were available, time series models could be used to predict price fluctuations.
*   **More Complex Models:** Explore deep learning models (e.g., neural networks) for potentially higher accuracy, especially with more complex features.
*   **External Data Integration:** Incorporate external data sources such as competitor pricing, seasonal trends, or economic indicators.
*   **A/B Testing:** For practical application, the model's predictions could be used to set dynamic pricing, and its effectiveness evaluated through A/B testing on the platform.

Overall, this notebook serves as a practical example of applying machine learning to real-world e-commerce data, offering insights and a robust predictive solution.